## Adapter notebook: generate ratemaps and manifold dataset from sequential optimization results

In [1]:

import numpy as np
import pickle
import json
import os
import jax.numpy as jnp
import jax

from sweep_analysis_sequential import compute_g_at_positions

# ── Configuration ──────────────────────────────────────────────────────────────
filepath = './data/251205/sweep_111115/run001'
counter  = 2          # which optimization stage (k)

POSITIONS_PATH = os.path.abspath('../Pilot Decoder/data/manifold/positions.npy')

adding_string = '_' if counter != '' else ''


In [2]:
# ── Load optimization results (mirrors PlotterSequential.ipynb) ────────────────
%cd "$filepath"

with open('parameters.json', 'r') as f:
    parameters = json.load(f)

with open(f'g0{adding_string}{counter}.pkl', 'rb') as f:
    g0 = pickle.load(f)

with open(f'om{adding_string}{counter}.pkl', 'rb') as f:
    om = pickle.load(f)

with open(f'S{adding_string}{counter}.pkl', 'rb') as f:
    S = pickle.load(f)

# Prefer final values
try:
    with open(f'g0_final{adding_string}{counter}.pkl', 'rb') as f:
        g0 = pickle.load(f)
except FileNotFoundError:
    pass

try:
    with open(f'S_final{adding_string}{counter}.pkl', 'rb') as f:
        S = pickle.load(f)
except FileNotFoundError:
    pass

D = g0.shape[0]
print(f'D={D}, g0={g0.shape}, om={om.shape}, S={S.shape}')

/home/julian/Projects/MasterThesis/ICLR_Actionable_Reps/data/251205/sweep_111115/run001
D=65, g0=(65,), om=(32, 2), S=(65, 65)


In [3]:

# ── Ratemaps ───────────────────────────────────────────────────────────────────
# Returns list of ratemaps, one per width; each entry has shape (D, res, res).

def get_ratemaps(g0, om, S, res, widths):
    """Compute ratemaps on square grids of different room sizes.

    Args:
        g0:     activity at origin, shape (D,)
        om:     frequencies, shape (M, 2)
        S:      change-of-basis matrix, shape (D, D)
        res:    grid resolution per side
        widths: iterable of absolute room widths (positions span [-w/2, w/2])

    Returns:
        List of arrays, each shape (D, res, res), one per width.
    """
    maps = []
    for w in widths:
        xs = np.linspace(-w / 2, w / 2, res)
        grid = np.meshgrid(xs, xs)          # each (res, res)
        phi = np.stack([grid[0].ravel(), grid[1].ravel()], axis=1)  # (res^2, 2)
        V = np.array(compute_g_at_positions(g0, om, S, phi))        # (D, res^2)
        maps.append(V.reshape(D, res, res))
    return maps


res    = 70
widths = (1, 2, 4)

Vs = get_ratemaps(g0, om, S, res, widths)
V_small, V_medium, V_large = Vs

print('V_small shape :', V_small.shape)   # spans [-0.5, 0.5]
print('V_medium shape:', V_medium.shape)  # spans [-1.0, 1.0]
print('V_large shape :', V_large.shape)   # spans [-2.0, 2.0]

# Save to disk
np.save(f'ratemaps_small_{counter}.npy',  V_small)
np.save(f'ratemaps_medium_{counter}.npy', V_medium)
np.save(f'ratemaps_large_{counter}.npy',  V_large)
print('Ratemaps saved.')


V_small shape : (65, 70, 70)
V_medium shape: (65, 70, 70)
V_large shape : (65, 70, 70)
Ratemaps saved.


In [4]:
# ── Manifold dataset ───────────────────────────────────────────────────────────
# Load absolute positions (B, L, 2), compute representation at each position,
# store as (B, L, D).

positions = np.load(POSITIONS_PATH)          # (B, L, 2)
B, L, _ = positions.shape
print(f'Positions: B={B}, L={L}')

# Flatten to (B*L, 2), compute representations, then reshape back
phi_flat = positions.reshape(B * L, 2)       # (B*L, 2)
G_flat = np.array(compute_g_at_positions(g0, om, S, phi_flat))  # (D, B*L)
representations = G_flat.T.reshape(B, L, D)  # (B, L, D)

print(f'Representations: {representations.shape}')  # (B, L, D)

np.save(f'representations_{counter}.npy', representations)
np.save(f'positions_manifold_{counter}.npy', positions)
print('Manifold dataset saved.')

Positions: B=100, L=1000
Representations: (100, 1000, 65)
Manifold dataset saved.
